In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from tabulate import tabulate
import matplotlib.pyplot as plt
import squarify

# Load and preprocess data
def load_and_preprocess_data(file_path):
    data = pd.read_csv(file_path)
    data['Brand_Name'].fillna('Unknown', inplace=True)
    data['Sales_Date'] = pd.to_datetime(data['Sales_Date'], errors='coerce')
    
    # Drop rows with invalid Sales_Date
    data = data.dropna(subset=['Sales_Date'])
    
    today = datetime.now()
    data['Recency'] = (today - data['Sales_Date']).dt.days
    data['Monetary_Value'] = pd.to_numeric(data['Monetary_Value'], errors='coerce')
    
    # Drop rows with invalid Monetary_Value
    data = data.dropna(subset=['Monetary_Value'])
    
    rfm = data.groupby('Client_ID').agg({
        'Recency': 'min',
        'Sales_Date': 'count',  # Frequency
        'Monetary_Value': 'sum'  # Monetary
    }).rename(columns={'Sales_Date': 'Frequency'}).reset_index()
    return rfm

# Assign RFM scores
def assign_rfm_scores(rfm):
    # Ensure all necessary columns are numeric
    rfm['Recency'] = pd.to_numeric(rfm['Recency'], errors='coerce').fillna(0)
    rfm['Frequency'] = pd.to_numeric(rfm['Frequency'], errors='coerce').fillna(0)
    rfm['Monetary_Value'] = pd.to_numeric(rfm['Monetary_Value'], errors='coerce').fillna(0)
    
    # Check for any remaining non-numeric values
    if rfm[['Recency', 'Frequency', 'Monetary_Value']].isnull().values.any():
        raise ValueError("Non-numeric values found in RFM columns after conversion.")

    # Calculate quantiles
    quantiles = rfm[['Recency', 'Frequency', 'Monetary_Value']].quantile(q=[0.2, 0.4, 0.6, 0.8]).to_dict()

    def r_score(x):
        if x <= quantiles['Recency'][0.2]:
            return 5
        elif x <= quantiles['Recency'][0.4]:
            return 4
        elif x <= quantiles['Recency'][0.6]:
            return 3
        elif x <= quantiles['Recency'][0.8]:
            return 2
        else:
            return 1

    def f_score(x):
        if x <= quantiles['Frequency'][0.2]:
            return 1
        elif x <= quantiles['Frequency'][0.4]:
            return 2
        elif x <= quantiles['Frequency'][0.6]:
            return 3
        elif x <= quantiles['Frequency'][0.8]:
            return 4
        else:
            return 5

    def m_score(x):
        if x <= quantiles['Monetary_Value'][0.2]:
            return 1
        elif x <= quantiles['Monetary_Value'][0.4]:
            return 2
        elif x <= quantiles['Monetary_Value'][0.6]:
            return 3
        elif x <= quantiles['Monetary_Value'][0.8]:
            return 4
        else:
            return 5

    rfm['R_Score'] = rfm['Recency'].apply(r_score)
    rfm['F_Score'] = rfm['Frequency'].apply(f_score)
    rfm['M_Score'] = rfm['Monetary_Value'].apply(m_score)
    rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
    
    # Sort by RFM_Score in decreasing order
    rfm = rfm.sort_values(by='RFM_Score', ascending=False).reset_index(drop=True)
    return rfm

# Normalize features
def normalize_features(rfm):
    scaler = MinMaxScaler()
    rfm[['Recency', 'Frequency', 'Monetary_Value']] = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary_Value']])
    return rfm

# Perform k-means clustering
def perform_clustering(rfm, n_clusters=5):
    features = rfm[['Recency', 'Frequency', 'Monetary_Value']]
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(features)
    rfm['Cluster'] = kmeans.labels_
    return rfm, kmeans

# Assign cluster names
def assign_cluster_names(rfm):
    cluster_labels = {
        0: "Potential",
        1: "Promising",
        2: "Can't-lose them",
        3: "At risk",
        4: "Lost"
    }

    rfm['Cluster'] = rfm['Cluster'].map(cluster_labels)
    return rfm

# Plot a treemap of the clusters
def plot_treemap(rfm):
    cluster_summary = rfm.groupby('Cluster').agg({
        'Client_ID': 'count',
        'R_Score': 'mean',
        'F_Score': 'mean',
        'M_Score': 'mean'
    }).rename(columns={'Client_ID': 'Count'}).reset_index()
    
    cluster_summary['Percentage'] = cluster_summary['Count'] / cluster_summary['Count'].sum() * 100
    labels = [
        f"{row['Cluster']}\n{row['Percentage']:.2f}%\nR: {row['R_Score']:.2f}\nF: {row['F_Score']:.2f}\nM: {row['M_Score']:.2f}"
        for index, row in cluster_summary.iterrows()
    ]
    
    sizes = cluster_summary['Count']
    plt.figure(figsize=(12, 8))
    squarify.plot(sizes=sizes, label=labels, alpha=0.8)
    plt.axis('off')
    plt.show()

# Calculate the silhouette score for model evaluation
def calculate_silhouette_score(rfm):
    features = rfm[['Recency', 'Frequency', 'Monetary_Value']]
    score = silhouette_score(features, rfm['Cluster'])
    return score

# Main function
def main(file_path):
    rfm = load_and_preprocess_data(file_path)
    rfm = assign_rfm_scores(rfm)
    rfm = normalize_features(rfm)
    rfm, kmeans = perform_clustering(rfm, n_clusters=5)
    rfm = assign_cluster_names(rfm)
    
    # Plot the treemap
    plot_treemap(rfm)
    
    # Print table with tabulate
    print(tabulate(rfm, headers='keys', tablefmt='pretty'))
    
    # Save the table to a CSV file
    rfm.to_csv('rfm_table.csv', index=False)
    
    # Calculate and print silhouette score
    silhouette_score = calculate_silhouette_score(rfm)
    print(f"Silhouette Score: {silhouette_score:.2f}")

# File path to the CSV file
file_path = "/Users/igormol/Desktop/Recsys_Models/samples/Marajo.csv"
main(file_path)


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from tabulate import tabulate
import matplotlib.pyplot as plt
import squarify

# Load and preprocess data
def load_and_preprocess_data(file_path):
    data = pd.read_csv(file_path)
    data['Brand_Name'].fillna('Unknown', inplace=True)
    data['Sales_Date'] = pd.to_datetime(data['Sales_Date'], errors='coerce')
    
    # Drop rows with invalid Sales_Date
    data = data.dropna(subset=['Sales_Date'])
    
    today = datetime.now()
    data['Recency'] = (today - data['Sales_Date']).dt.days
    data['Monetary_Value'] = pd.to_numeric(data['Monetary_Value'], errors='coerce')
    
    # Drop rows with invalid Monetary_Value
    data = data.dropna(subset=['Monetary_Value'])
    
    rfm = data.groupby('Client_ID').agg({
        'Recency': 'min',
        'Sales_Date': 'count',  # Frequency
        'Monetary_Value': 'sum'  # Monetary
    }).rename(columns={'Sales_Date': 'Frequency'}).reset_index()
    return rfm

# Assign RFM scores
def assign_rfm_scores(rfm):
    # Ensure all necessary columns are numeric
    rfm['Recency'] = pd.to_numeric(rfm['Recency'], errors='coerce').fillna(0)
    rfm['Frequency'] = pd.to_numeric(rfm['Frequency'], errors='coerce').fillna(0)
    rfm['Monetary_Value'] = pd.to_numeric(rfm['Monetary_Value'], errors='coerce').fillna(0)
    
    # Check for any remaining non-numeric values
    if rfm[['Recency', 'Frequency', 'Monetary_Value']].isnull().values.any():
        raise ValueError("Non-numeric values found in RFM columns after conversion.")

    # Calculate quantiles
    quantiles = rfm[['Recency', 'Frequency', 'Monetary_Value']].quantile(q=[0.2, 0.4, 0.6, 0.8]).to_dict()

    def r_score(x):
        if x <= quantiles['Recency'][0.2]:
            return 5
        elif x <= quantiles['Recency'][0.4]:
            return 4
        elif x <= quantiles['Recency'][0.6]:
            return 3
        elif x <= quantiles['Recency'][0.8]:
            return 2
        else:
            return 1

    def f_score(x):
        if x <= quantiles['Frequency'][0.2]:
            return 1
        elif x <= quantiles['Frequency'][0.4]:
            return 2
        elif x <= quantiles['Frequency'][0.6]:
            return 3
        elif x <= quantiles['Frequency'][0.8]:
            return 4
        else:
            return 5

    def m_score(x):
        if x <= quantiles['Monetary_Value'][0.2]:
            return 1
        elif x <= quantiles['Monetary_Value'][0.4]:
            return 2
        elif x <= quantiles['Monetary_Value'][0.6]:
            return 3
        elif x <= quantiles['Monetary_Value'][0.8]:
            return 4
        else:
            return 5

    rfm['R_Score'] = rfm['Recency'].apply(r_score)
    rfm['F_Score'] = rfm['Frequency'].apply(f_score)
    rfm['M_Score'] = rfm['Monetary_Value'].apply(m_score)
    rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
    
    # Sort by RFM_Score in decreasing order
    rfm = rfm.sort_values(by='RFM_Score', ascending=False).reset_index(drop=True)
    return rfm

# Normalize features (RFM Scores)
def normalize_features(rfm):
    scaler = MinMaxScaler()
    rfm[['R_Score', 'F_Score', 'M_Score']] = scaler.fit_transform(rfm[['R_Score', 'F_Score', 'M_Score']])
    return rfm

# Perform k-means clustering
def perform_clustering(rfm, n_clusters=5):
    features = rfm[['R_Score', 'F_Score', 'M_Score']]
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(features)
    rfm['Cluster'] = kmeans.labels_
    return rfm, kmeans

# Assign cluster names
def assign_cluster_names(rfm):
    cluster_labels = {
        0: "Potential",
        1: "Promising",
        2: "Can't-lose them",
        3: "At risk",
        4: "Lost"
    }

    rfm['Cluster'] = rfm['Cluster'].map(cluster_labels)
    return rfm

# Plot a treemap of the clusters
def plot_treemap(rfm):
    cluster_summary = rfm.groupby('Cluster').agg({
        'Client_ID': 'count',
        'R_Score': 'mean',
        'F_Score': 'mean',
        'M_Score': 'mean'
    }).rename(columns={'Client_ID': 'Count'}).reset_index()
    
    cluster_summary['Percentage'] = cluster_summary['Count'] / cluster_summary['Count'].sum() * 100
    labels = [
        f"{row['Cluster']}\n{row['Percentage']:.2f}%\nR: {row['R_Score']:.2f}\nF: {row['F_Score']:.2f}\nM: {row['M_Score']:.2f}"
        for index, row in cluster_summary.iterrows()
    ]
    
    sizes = cluster_summary['Count']
    plt.figure(figsize=(12, 8))
    squarify.plot(sizes=sizes, label=labels, alpha=0.8)
    plt.axis('off')
    plt.show()

# Calculate the silhouette score for model evaluation
def calculate_silhouette_score(rfm):
    features = rfm[['R_Score', 'F_Score', 'M_Score']]
    score = silhouette_score(features, rfm['Cluster'])
    return score

# Main function
def main(file_path):
    rfm = load_and_preprocess_data(file_path)
    rfm = assign_rfm_scores(rfm)
    rfm = normalize_features(rfm)
    rfm, kmeans = perform_clustering(rfm, n_clusters=5)
    rfm = assign_cluster_names(rfm)
    
    # Plot the treemap
    plot_treemap(rfm)
    
    # Print table with tabulate
    print(tabulate(rfm, headers='keys', tablefmt='pretty'))
    
    # Save the table to a CSV file
    rfm.to_csv('rfm_table.csv', index=False)
    
    # Calculate and print silhouette score
    silhouette_score = calculate_silhouette_score(rfm)
    print(f"Silhouette Score: {silhouette_score:.2f}")

# File path to the CSV file
file_path = "/Users/igormol/Desktop/Recsys_Models/samples/Marajo.csv"
main(file_path)


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from tabulate import tabulate
import matplotlib.pyplot as plt
import squarify

# Load and preprocess data
def load_and_preprocess_data(file_path):
    data = pd.read_csv(file_path)
    data['Brand_Name'].fillna('Unknown', inplace=True)
    data['Sales_Date'] = pd.to_datetime(data['Sales_Date'], errors='coerce')
    
    # Drop rows with invalid Sales_Date
    data = data.dropna(subset=['Sales_Date'])
    
    today = datetime.now()
    data['Recency'] = (today - data['Sales_Date']).dt.days
    data['Monetary_Value'] = pd.to_numeric(data['Monetary_Value'], errors='coerce')
    
    # Drop rows with invalid Monetary_Value
    data = data.dropna(subset=['Monetary_Value'])
    
    rfm = data.groupby('Client_ID').agg({
        'Recency': 'min',
        'Sales_Date': 'count',  # Frequency
        'Monetary_Value': 'sum'  # Monetary
    }).rename(columns={'Sales_Date': 'Frequency'}).reset_index()
    return rfm

# Assign RFM scores
def assign_rfm_scores(rfm):
    # Ensure all necessary columns are numeric
    rfm['Recency'] = pd.to_numeric(rfm['Recency'], errors='coerce').fillna(0)
    rfm['Frequency'] = pd.to_numeric(rfm['Frequency'], errors='coerce').fillna(0)
    rfm['Monetary_Value'] = pd.to_numeric(rfm['Monetary_Value'], errors='coerce').fillna(0)
    
    # Check for any remaining non-numeric values
    if rfm[['Recency', 'Frequency', 'Monetary_Value']].isnull().values.any():
        raise ValueError("Non-numeric values found in RFM columns after conversion.")

    # Calculate quantiles
    quantiles = rfm[['Recency', 'Frequency', 'Monetary_Value']].quantile(q=[0.2, 0.4, 0.6, 0.8]).to_dict()

    def r_score(x):
        if x <= quantiles['Recency'][0.2]:
            return 5
        elif x <= quantiles['Recency'][0.4]:
            return 4
        elif x <= quantiles['Recency'][0.6]:
            return 3
        elif x <= quantiles['Recency'][0.8]:
            return 2
        else:
            return 1

    def f_score(x):
        if x <= quantiles['Frequency'][0.2]:
            return 1
        elif x <= quantiles['Frequency'][0.4]:
            return 2
        elif x <= quantiles['Frequency'][0.6]:
            return 3
        elif x <= quantiles['Frequency'][0.8]:
            return 4
        else:
            return 5

    def m_score(x):
        if x <= quantiles['Monetary_Value'][0.2]:
            return 1
        elif x <= quantiles['Monetary_Value'][0.4]:
            return 2
        elif x <= quantiles['Monetary_Value'][0.6]:
            return 3
        elif x <= quantiles['Monetary_Value'][0.8]:
            return 4
        else:
            return 5

    rfm['R_Score'] = rfm['Recency'].apply(r_score)
    rfm['F_Score'] = rfm['Frequency'].apply(f_score)
    rfm['M_Score'] = rfm['Monetary_Value'].apply(m_score)
    rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
    
    # Sort by RFM_Score in decreasing order
    rfm = rfm.sort_values(by='RFM_Score', ascending=False).reset_index(drop=True)
    return rfm

# Normalize features (RFM Scores)
def normalize_features(rfm):
    scaler = MinMaxScaler()
    rfm[['R_Score', 'F_Score', 'M_Score']] = scaler.fit_transform(rfm[['R_Score', 'F_Score', 'M_Score']])
    return rfm

# Perform k-means clustering
def perform_clustering(rfm, n_clusters=5):
    features = rfm[['R_Score', 'F_Score', 'M_Score']]
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(features)
    rfm['Cluster'] = kmeans.labels_
    return rfm, kmeans

# Assign cluster names
def assign_cluster_names(rfm):
    cluster_labels = {
        0: "Potential",
        1: "Promising",
        2: "Can't-lose them",
        3: "At risk",
        4: "Lost"
    }

    rfm['Cluster'] = rfm['Cluster'].map(cluster_labels)
    return rfm

# Plot a treemap of the clusters
def plot_treemap(rfm):
    cluster_summary = rfm.groupby('Cluster').agg({
        'Client_ID': 'count',
        'R_Score': 'mean',
        'F_Score': 'mean',
        'M_Score': 'mean'
    }).rename(columns={'Client_ID': 'Count'}).reset_index()
    
    cluster_summary['Percentage'] = cluster_summary['Count'] / cluster_summary['Count'].sum() * 100
    labels = [
        f"{row['Cluster']}\n{row['Percentage']:.2f}%\nR: {row['R_Score']:.2f}\nF: {row['F_Score']:.2f}\nM: {row['M_Score']:.2f}"
        for index, row in cluster_summary.iterrows()
    ]
    
    sizes = cluster_summary['Count']
    plt.figure(figsize=(12, 8))
    squarify.plot(sizes=sizes, label=labels, alpha=0.8)
    plt.axis('off')
    plt.show()

# Calculate the silhouette score for each client
def calculate_silhouette_scores(rfm):
    features = rfm[['R_Score', 'F_Score', 'M_Score']]
    rfm['Silhouette_Score'] = silhouette_samples(features, rfm['Cluster'])
    return rfm

# Main function
def main(file_path):
    rfm = load_and_preprocess_data(file_path)
    rfm = assign_rfm_scores(rfm)
    rfm = normalize_features(rfm)
    rfm, kmeans = perform_clustering(rfm, n_clusters=5)
    rfm = assign_cluster_names(rfm)
    rfm = calculate_silhouette_scores(rfm)
    
    # Plot the treemap
    plot_treemap(rfm)
    
    # Print table with tabulate
    print(tabulate(rfm, headers='keys', tablefmt='pretty'))
    
    # Save the table to a CSV file
    rfm.to_csv('rfm_table.csv', index=False)
    
    # Print average silhouette score
    avg_silhouette_score = rfm['Silhouette_Score'].mean()
    print(f"Average Silhouette Score: {avg_silhouette_score:.2f}")

# File path to the CSV file
file_path = "/Users/igormol/Desktop/Recsys_Models/samples/Marajo.csv"
main(file_path)
